## Notebook 02: Data Cleaning

### Objective
This notebook focuses on cleaning the bank marketing dataset by handling missing values, correcting data types, removing duplicates, and preparing the data for analysis and modeling.

### Outcomes
- Handle missing values appropriately
- Convert data types to correct formats
- Remove duplicate records
- Identify and handle outliers
- Save the cleaned dataset for further analysis

### Steps Covered
1. Handle missing values
2. Convert data types
3. Remove duplicates
4. Handle outliers in numerical columns
5. Save cleaned dataset

### Importing libraries and loading the dataset

In [1]:
### Importing libraries and loading the dataset

import pandas as pd
import numpy as np
import os

RAW_DATA_PATH = "../data/bank-full.csv"
CLEANED_DATA_PATH = "../outputs/cleaned_data/bank_cleaned.csv"

print("-"*140)
print("LOADING THE DATASET")
print("-"*140)

df = pd.read_csv(RAW_DATA_PATH, sep=';')

print(f"Dataset shape before cleaning: {df.shape[0]} rows and {df.shape[1]} columns")
print(f"First 3 rows:")
print(df.head(3))

--------------------------------------------------------------------------------------------------------------------------------------------
LOADING THE DATASET
--------------------------------------------------------------------------------------------------------------------------------------------
Dataset shape before cleaning: 45211 rows and 17 columns
First 3 rows:
   age           job  marital  education default  balance housing loan  \
0   58    management  married   tertiary      no     2143     yes   no   
1   44    technician   single  secondary      no       29     yes   no   
2   33  entrepreneur  married  secondary      no        2     yes  yes   

   contact  day month  duration  campaign  pdays  previous poutcome   y  
0  unknown    5   may       261         1     -1         0  unknown  no  
1  unknown    5   may       151         1     -1         0  unknown  no  
2  unknown    5   may        76         1     -1         0  unknown  no  


### Handling missing values

In [2]:
### Handling missing values

print("-"*140)
print("HANDLING MISSING VALUES")
print("-"*140)

# Check missing values before handling
missing_before = df.isnull().sum()
print("Missing values before cleaning:")
print(missing_before[missing_before > 0] if any(missing_before > 0) else "No missing values found.")

# The dataset has no missing values as seen in the report
# But we check for 'unknown' values in categorical columns that need attention
print("\nChecking for 'unknown' values in categorical columns:")
categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'poutcome']

for col in categorical_cols:
    unknown_count = (df[col] == 'unknown').sum()
    if unknown_count > 0:
        print(f"  {col}: {unknown_count} unknown values ({unknown_count/len(df)*100:.2f}%)")

print("\nNo missing values (NaN) found in the dataset.")
print("'Unknown' values will be handled during feature engineering.")

--------------------------------------------------------------------------------------------------------------------------------------------
HANDLING MISSING VALUES
--------------------------------------------------------------------------------------------------------------------------------------------
Missing values before cleaning:
No missing values found.

Checking for 'unknown' values in categorical columns:
  job: 288 unknown values (0.64%)
  education: 1857 unknown values (4.11%)
  contact: 13020 unknown values (28.80%)
  poutcome: 36959 unknown values (81.75%)

No missing values (NaN) found in the dataset.
'Unknown' values will be handled during feature engineering.


### Converting data types

In [3]:
### Converting data types

print("-"*140)
print("CONVERTING DATA TYPES")
print("-"*140)

print("Data types before conversion:")
print(df.dtypes)

# Convert age to int (already int, but ensure)
df['age'] = df['age'].astype(int)

# Convert balance to int
df['balance'] = df['balance'].astype(int)

# Convert day to int
df['day'] = df['day'].astype(int)

# Convert duration to int
df['duration'] = df['duration'].astype(int)

# Convert campaign to int
df['campaign'] = df['campaign'].astype(int)

# Convert pdays to int
df['pdays'] = df['pdays'].astype(int)

# Convert previous to int
df['previous'] = df['previous'].astype(int)

# Convert target variable y to binary (yes=1, no=0)
df['y_binary'] = df['y'].map({'yes': 1, 'no': 0})

print("\nData types after conversion:")
print(df.dtypes)

print("\nTarget variable distribution (binary):")
print(df['y_binary'].value_counts())
print(f"Positive rate: {df['y_binary'].mean()*100:.2f}%")

--------------------------------------------------------------------------------------------------------------------------------------------
CONVERTING DATA TYPES
--------------------------------------------------------------------------------------------------------------------------------------------
Data types before conversion:
age          int64
job            str
marital        str
education      str
default        str
balance      int64
housing        str
loan           str
contact        str
day          int64
month          str
duration     int64
campaign     int64
pdays        int64
previous     int64
poutcome       str
y              str
dtype: object

Data types after conversion:
age          int32
job            str
marital        str
education      str
default        str
balance      int32
housing        str
loan           str
contact        str
day          int32
month          str
duration     int32
campaign     int32
pdays        int32
previous     int32
poutcome      

### Removing duplicate rows

In [4]:
### Removing duplicate rows

print("-"*140)
print("REMOVING DUPLICATE ROWS")
print("-"*140)

duplicate_count = df.duplicated().sum()
print(f"Number of duplicate rows before removal: {duplicate_count}")

if duplicate_count > 0:
    df = df.drop_duplicates()
    print(f"Number of duplicate rows after removal: {df.duplicated().sum()}")
    print(f"Dataset shape after removing duplicates: {df.shape[0]} rows and {df.shape[1]} columns")
else:
    print("No duplicate rows found. Dataset unchanged.")

--------------------------------------------------------------------------------------------------------------------------------------------
REMOVING DUPLICATE ROWS
--------------------------------------------------------------------------------------------------------------------------------------------
Number of duplicate rows before removal: 0
No duplicate rows found. Dataset unchanged.


### Handling unknown values in categorical columns

In [5]:
### Handling unknown values in categorical columns

print("-"*140)
print("HANDLING UNKNOWN VALUES IN CATEGORICAL COLUMNS")
print("-"*140)

categorical_cols = ['job', 'education', 'contact', 'poutcome']

print("Unknown value counts before handling:")
for col in categorical_cols:
    unknown_count = (df[col] == 'unknown').sum()
    unknown_percent = (unknown_count / len(df)) * 100
    print(f"  {col}: {unknown_count} unknown ({unknown_percent:.2f}%)")

print("\nStrategy: Replace 'unknown' with the mode (most frequent value) for each column")

for col in categorical_cols:
    mode_value = df[col][df[col] != 'unknown'].mode()[0]
    unknown_mask = df[col] == 'unknown'
    df.loc[unknown_mask, col] = mode_value
    print(f"\n{col}: Replaced 'unknown' with '{mode_value}'")

print("\nUnknown value counts after handling:")
for col in categorical_cols:
    unknown_count = (df[col] == 'unknown').sum()
    print(f"  {col}: {unknown_count} unknown")

--------------------------------------------------------------------------------------------------------------------------------------------
HANDLING UNKNOWN VALUES IN CATEGORICAL COLUMNS
--------------------------------------------------------------------------------------------------------------------------------------------
Unknown value counts before handling:
  job: 288 unknown (0.64%)
  education: 1857 unknown (4.11%)
  contact: 13020 unknown (28.80%)
  poutcome: 36959 unknown (81.75%)

Strategy: Replace 'unknown' with the mode (most frequent value) for each column

job: Replaced 'unknown' with 'blue-collar'

education: Replaced 'unknown' with 'secondary'

contact: Replaced 'unknown' with 'cellular'

poutcome: Replaced 'unknown' with 'failure'

Unknown value counts after handling:
  job: 0 unknown
  education: 0 unknown
  contact: 0 unknown
  poutcome: 0 unknown


### Handling outliers in numerical columns

In [6]:
### Handling outliers in numerical columns

print("-"*140)
print("HANDLING OUTLIERS IN NUMERICAL COLUMNS")
print("-"*140)

numeric_cols = ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous']

outlier_summary = []

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outlier_count = len(outliers)
    outlier_percent = (outlier_count / len(df)) * 100
    
    outlier_summary.append({
        'Column': col,
        'Lower Bound': round(lower_bound, 2),
        'Upper Bound': round(upper_bound, 2),
        'Outliers Count': outlier_count,
        'Outliers Percentage': round(outlier_percent, 2)
    })
    
    print(f"\n{col.upper()}:")
    print(f"  Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
    print(f"  Outliers: {outlier_count} ({outlier_percent:.2f}%)")
    print(f"  Range: {df[col].min()} to {df[col].max()}")

print("\n" + "-"*140)
print("OUTLIER SUMMARY TABLE")
print("-"*140)
outlier_df = pd.DataFrame(outlier_summary)
print(outlier_df.to_string())

print("\nNote: Outliers will be kept for now as they may contain valuable information.")
print("They will be handled during feature engineering if necessary.")

--------------------------------------------------------------------------------------------------------------------------------------------
HANDLING OUTLIERS IN NUMERICAL COLUMNS
--------------------------------------------------------------------------------------------------------------------------------------------

AGE:
  Q1: 33.0, Q3: 48.0, IQR: 15.0
  Outliers: 487 (1.08%)
  Range: 18 to 95

BALANCE:
  Q1: 72.0, Q3: 1428.0, IQR: 1356.0
  Outliers: 4729 (10.46%)
  Range: -8019 to 102127

DURATION:
  Q1: 103.0, Q3: 319.0, IQR: 216.0
  Outliers: 3235 (7.16%)
  Range: 0 to 4918

CAMPAIGN:
  Q1: 1.0, Q3: 3.0, IQR: 2.0
  Outliers: 3064 (6.78%)
  Range: 1 to 63

PDAYS:
  Q1: -1.0, Q3: -1.0, IQR: 0.0
  Outliers: 8257 (18.26%)
  Range: -1 to 871

PREVIOUS:
  Q1: 0.0, Q3: 0.0, IQR: 0.0
  Outliers: 8257 (18.26%)
  Range: 0 to 275

--------------------------------------------------------------------------------------------------------------------------------------------
OUTLIER SUMMARY TABL

### Verifying data quality after cleaning

In [7]:
### Verifying data quality after cleaning

print("-"*140)
print("VERIFYING DATA QUALITY AFTER CLEANING")
print("-"*140)

print("Dataset shape:", df.shape)
print("\nData types:")
print(df.dtypes)

print("\nMissing values check:")
missing_after = df.isnull().sum()
print(missing_after[missing_after > 0] if any(missing_after > 0) else "No missing values found.")

print("\nTarget variable distribution:")
print(df['y'].value_counts())
print(f"\nTarget as binary: {df['y_binary'].value_counts()}")

print("\nSample of cleaned data (first 5 rows):")
print(df.head())

--------------------------------------------------------------------------------------------------------------------------------------------
VERIFYING DATA QUALITY AFTER CLEANING
--------------------------------------------------------------------------------------------------------------------------------------------
Dataset shape: (45211, 18)

Data types:
age          int32
job            str
marital        str
education      str
default        str
balance      int32
housing        str
loan           str
contact        str
day          int32
month          str
duration     int32
campaign     int32
pdays        int32
previous     int32
poutcome       str
y              str
y_binary     int64
dtype: object

Missing values check:
No missing values found.

Target variable distribution:
y
no     39922
yes     5289
Name: count, dtype: int64

Target as binary: y_binary
0    39922
1     5289
Name: count, dtype: int64

Sample of cleaned data (first 5 rows):
   age           job  marital  educ

### Saving the cleaned dataset

In [8]:
### Saving the cleaned dataset

import os

print("-"*140)
print("SAVING THE CLEANED DATASET")
print("-"*140)

os.makedirs("../outputs/cleaned_data", exist_ok=True)

df.to_csv(CLEANED_DATA_PATH, index=False)

print(f"Cleaned dataset saved to: {CLEANED_DATA_PATH}")
print(f"Final dataset shape: {df.shape[0]} rows and {df.shape[1]} columns")
print(f"Columns in cleaned dataset: {list(df.columns)}")

print("\n" + "-"*140)
print("NOTEBOOK 02 COMPLETED SUCCESSFULLY")
print("-"*140)
print("\nData cleaning completed.")
print("Proceed to Notebook 03 for data analysis.")

--------------------------------------------------------------------------------------------------------------------------------------------
SAVING THE CLEANED DATASET
--------------------------------------------------------------------------------------------------------------------------------------------
Cleaned dataset saved to: ../outputs/cleaned_data/bank_cleaned.csv
Final dataset shape: 45211 rows and 18 columns
Columns in cleaned dataset: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y', 'y_binary']

--------------------------------------------------------------------------------------------------------------------------------------------
NOTEBOOK 02 COMPLETED SUCCESSFULLY
--------------------------------------------------------------------------------------------------------------------------------------------

Data cleaning completed.
Proceed to Notebook 03 f